# Plots - Juntar precição de todos os modelos para análise comparativa
Autor:  Viviane da Rosa Sommer

Atualizado: 12/09/2025

Objetivo: Gerar plots comparativos para análise visual e quantitativa dos resultados de classificação, segmentação e respostas do modelo LLM, consolidando as informações de todos os modelos em uma única visualização para facilitar a avaliação de desempenho.

## Importações necessárias

In [ ]:
import os
import cv2
import json
import numpy as np
import textwrap
import matplotlib
import gc
import matplotlib.pyplot as plt
import tqdm
from skimage import measure
from typing import Tuple, Optional
matplotlib.use('Agg')

## Declaração de Constantes e Modelos

In [ ]:
operations = [d for d in os.listdir("fase1-0-7-3/validas") if os.path.isdir(os.path.join("fase1-0-7-3/validas", d))]
PATH_VALID_IMAGES = "fase1-0-7-3/validas/"
PATH_CLASSIFICATION = "fase2-0-7-3/"
PATH_LLM_JSONS = "fase3-0-7-3/"
PATH_OUTPUT = "fase4-0-7-3/"

## Funções necessárias

In [ ]:
def process_operation(op_name: str) -> None:
    """
    Processes images for a given operation, generating output visualizations with classification and LLM responses.

    Args:
        op_name (str): The name of the operation to process.

    This function:
        - Iterates over 'positive' and 'negative' class directories.
        - For each image, checks for corresponding classification and LLM JSON files.
        - Reads the image, mask (if available), and metadata.
        - Generates a visualization with the original image, segmentation contour, and classification/LLM info.
        - Saves the output image to the output directory.
    """
    img_dir = PATH_VALID_IMAGES + f"{op_name}"
    llm_dir = PATH_LLM_JSONS + f"{op_name}"
    out_dir = PATH_OUTPUT + f"{op_name}"
    os.makedirs(out_dir, exist_ok=True)
    for class_dir in ['positive', 'negative']:
        data_dir = PATH_CLASSIFICATION + f"{op_name}/{class_dir}"
        if not os.path.exists(data_dir):
            continue
        img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png'))]
        for img_file in img_files:
            output_path = os.path.join(out_dir, img_file.split('.')[0] + '.png')
            if os.path.exists(output_path):
                continue
            json_path = os.path.join(data_dir, os.path.splitext(img_file)[0] + '.json')
            if not os.path.exists(json_path):
                continue
            img_path = os.path.join(img_dir, img_file)
            result = read_image(img_path)
            if result is None:
                continue
            img, image = result
            mask = None
            if class_dir == 'positive':
                mask_path = os.path.join(data_dir, os.path.splitext(img_file)[0] + '_mask.png')
                if os.path.exists(mask_path):
                    mask = cv2.imread(mask_path, 0)
            llm_json_path = os.path.join(llm_dir, os.path.splitext(img_file)[0] + '.json')
            if os.path.exists(llm_json_path):
                with open(llm_json_path, 'r') as f:
                    llm_data = json.load(f)
                llm_response = llm_data.get('resposta', '')
            else:
                llm_response = ''
            with open(json_path, 'r') as f:
                data = json.load(f)
            class_label = data.get('classe', 'N/A')
            prob = data.get('probabilidade_classificacao', 0)
            if class_label == 'Positive':
                class_label = 'Positivo'
            elif class_label == 'Negative':
                class_label = 'Negativo'
            wrapped_response = "\n".join(textwrap.wrap(llm_response, width=60))
            fig, axs = plt.subplots(1, 3, figsize=(15, 5))
            fig.suptitle(img_file.split(".")[0], fontsize=16, fontweight='bold', y=0.92)
            axs[0].imshow(image)
            axs[0].set_title('Original Image')
            axs[0].axis('off')
            axs[1].imshow(img)
            if mask is not None and np.any(mask):
                contours = measure.find_contours(mask, 0.5)
                for contour in contours:
                    axs[1].plot(contour[:, 1], contour[:, 0], color='#FF00FF', linewidth=3)
                axs[1].set_title('Segmentation Contour')
            else:
                axs[1].set_title('Segmentation Contour\n')
                axs[1].text(0.5, 0.5, 'No mask', color='red', fontsize=16, ha='center', va='center', transform=axs[1].transAxes)
            axs[1].axis('off')
            axs[2].axis('off')
            text = f"Classification: {class_label}\nProbability: {prob*100:.1f}%\n\nChatGPT - Specialist:\n{wrapped_response}"
            axs[2].text(0.5, 0.5, text, fontsize=12, va='center', ha='center', multialignment='center', transform=axs[2].transAxes)
            plt.tight_layout(rect=[0, 0, 1, 0.88])
            plt.savefig(output_path)
            plt.clf()
            plt.close(fig)
            plt.close('all')
            gc.collect()


def read_image(filename: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """
    Reads an image from a file, crops it vertically, and returns both the cropped and original images.

    Args:
        filename (str): Path to the image file.

    Returns:
        Optional[Tuple[np.ndarray, np.ndarray]]: A tuple (cropped_image, original_image) if successful, else None.
    """
    try:
        image = cv2.imread(filename)
        if image is None:
            return None
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width, _ = image.shape
        mid = height // 2
        top = max(0, mid - int(0.34 * height))
        bottom = min(height, mid + int(0.34 * height))
        cropped_image = image[top:bottom, :]
        return cropped_image.astype(np.uint8), image
    except Exception:
        return None

## Gerar plots

In [ ]:
for op in tqdm(operations, desc="Processing operations"):
    process_operation(op)